The following file is used to create the results from section 4.1 in the paper. We train 4 metanetworks, one for each dataset and evaluate performance on the validation set.

In [ ]:
DATASETS = ['mnist', 'fashion_mnist', 'cifar10', 'svhn_cropped']
DATASET = DATASETS[0]

In [ ]:
from cnn_surgery.utils.load_dataset import load_multi_stage_dataset, load_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens, mse_mae
import pickle
N = 10  # number of metanetworks to train per dataset

In [ ]:
results = {}
for dataset in DATASETS:
    results[dataset] = []
    for i in range(N):  # Train N metanetworks per dataset
        data = load_multi_stage_dataset(dataset=dataset)
        weights_train, accuracies_train, config_train = data['train']
        weights_val, accuracies_val, config_val = data['val']
        meta_network, metrics = get_regressor_lens(weights_train, accuracies_train, weights_val, accuracies_val, device='cpu', verbose=True, return_metrics=True)
        ((mse_train, mae_train), (mse_val, mae_val), r2) = metrics
        results[dataset].append({
            'mse_train': mse_train,
            'mae_train': mae_train,
            'mse_val': mse_val,
            'mae_val': mae_val,
            'r2': r2
        })
        # Save each metanetwork for later use
        pickle.dump(meta_network, open(f'meta_network_{dataset}_{i}.pkl', 'wb'))

In [ ]:
import pandas as pd
print(pd.DataFrame(results).T.to_latex(float_format="%.3f"))